In [3]:
#this scripts uses calculated thermo cac equilibrium data to train ML models to predict D_max for different compositions. This uses CBFV as additional features.


In [13]:
#Import necessary libraries
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from CBFV import composition
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import re
from ast import literal_eval
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
from ax.service.ax_client import AxClient, ObjectiveProperties
from ax.core.arm import Arm
import ujson as js
from scipy.stats import sem

In [2]:
#according to the single tree notebook the best temperature ranges to utalize are: 2200, 1700, 2450, 2100, 2400, 1900, 1850
#according to the same notebook the NF or phase fraction features are ineffective at predicting D_max and thus will not be used

In [3]:
#pull training data
train_opt_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_train_opt.csv")

#pull test data
test_CALPHAD_df = pd.read_csv(r"Data/F(Composition)_Data/CALPHAD_alloys_test.csv")


train_opt_CALPHAD_df.head()

,alloy_string,DF_AG2CA_T0C,DF_AG2CA_T1000C,DF_AG2CA_T100C,DF_AG2CA_T1050C,DF_AG2CA_T1100C,DF_AG2CA_T1150C,DF_AG2CA_T1200C,DF_AG2CA_T1250C,DF_AG2CA_T1300C,...,NF_ZRSI_T50C,NF_ZRSI_T550C,NF_ZRSI_T600C,NF_ZRSI_T650C,NF_ZRSI_T700C,NF_ZRSI_T750C,NF_ZRSI_T800C,NF_ZRSI_T850C,NF_ZRSI_T900C,NF_ZRSI_T950C
0,B22.00Co4.00Fe68.00Y6.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Al10.00Ce60.00Cu20.00Ni10.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
#create the CBFV features for the training and test data
#start by pulling the formula column
train_opt_formula_df = pd.DataFrame({'formula': train_opt_CALPHAD_df['alloy_string']})
test_formula_df = pd.DataFrame({'formula': test_CALPHAD_df['alloy_string']})

#add a target column for CBFV api
train_opt_formula_df['target'] = 0
test_formula_df['target'] = 0

#use the CBFV api to create the features
train_opt_CBFV_df, _, train_opt_formulae, skipped_train = composition.generate_features(train_opt_formula_df, elem_prop='magpie')
test_CBFV_df, _, test_formulae, skipped_test = composition.generate_features(test_formula_df, elem_prop='magpie')


print(f"CBFV train features shape: {train_opt_CBFV_df.shape}, test features shape: {test_CBFV_df.shape}")
print(f"Number of skipped formulas: {len(skipped_train)}, {len(skipped_test)}")
train_opt_CBFV_df.head()


Processing Input Data: 100%|██████████| 882/882 [00:00<00:00, 27555.87it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 882/882 [00:00<00:00, 27128.69it/s]


	Creating Pandas Objects...


Processing Input Data: 100%|██████████| 98/98 [00:00<00:00, 49026.93it/s]


	Featurizing Compositions...


Assigning Features...: 100%|██████████| 98/98 [00:00<00:00, 28005.85it/s]

	Creating Pandas Objects...
CBFV train features shape: (882, 132), test features shape: (98, 132)
Number of skipped formulas: 0, 0


,avg_Number,avg_MendeleevNumber,avg_AtomicWeight,avg_MeltingT,avg_Column,avg_Row,avg_CovalentRadius,avg_Electronegativity,avg_NsValence,avg_NpValence,...,mode_NValence,mode_NsUnfilled,mode_NpUnfilled,mode_NdUnfilled,mode_NfUnfilled,mode_NUnfilled,mode_GSvolume_pa,mode_GSbandgap,mode_GSmagmom,mode_SpaceGroupNumber
0,22.200000,56.280000,48.044699,1926.70000,8.840000,3.620000,124.680000,1.841600,2.000000,0.220000,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
1,27.780000,55.100000,60.858759,1736.29190,7.980000,4.100000,143.910000,1.749000,1.420000,0.020000,...,11.0,1.0,0.0,0.0,0.0,1.0,11.070,0.0,0.000000,225.0
2,24.359036,58.662966,53.217519,2293.90019,8.946795,3.734973,123.841484,1.995602,1.859986,0.360036,...,8.0,0.0,0.0,4.0,0.0,4.0,10.730,0.0,2.110663,229.0
3,29.000000,53.040000,65.476074,1923.65240,4.760000,4.080000,146.560000,1.517600,1.800000,0.000000,...,4.0,0.0,0.0,8.0,0.0,8.0,23.195,0.0,0.000000,194.0
4,44.700000,35.200000,105.346294,1180.30100,6.300000,5.100000,173.300000,1.404000,1.800000,0.100000,...,4.0,0.0,0.0,9.0,13.0,22.0,37.240,0.0,0.000000,194.0


In [5]:
#filter the CALPHAD data to include only driving forces at the temepratures of interests
# Define the temperature ranges of interest
temperature_range = [2200, 1700, 2450, 2100, 2400, 1900, 1850]
temperature_range_str = [str(temp) for temp in temperature_range]

# Filter the columns to include only those with the specified temperature ranges and Df
filtered_CALPHAD_cols = [col for col in train_opt_CALPHAD_df.columns if any(temp in col for temp in temperature_range_str) and 'DF' in col]

print(f"Number of filtered CALPHAD columns: {len(filtered_CALPHAD_cols)}")

Number of filtered CALPHAD columns: 5628


In [ ]:
#combine the CBFV features with the filtered CALPHAD features for training and test data
X_train_opt = pd.concat([train_opt_CBFV_df, train_opt_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)
X_test = pd.concat([test_CBFV_df, test_CALPHAD_df[filtered_CALPHAD_cols]], axis=1)

print(f"Combined train features shape: {X_train_opt.shape}, Combined test features shape: {X_test.shape}")


Combined train features shape: (882, 5760), Combined test features shape: (98, 5760)


In [9]:
#load the dmax json data and create y data
D_max_dict = js.load(open(r"Data\dmax_data.json", 'r'))

y_train_opt = [D_max_dict[formula] for formula in train_opt_CALPHAD_df['alloy_string']]
y_test = [D_max_dict[formula] for formula in test_CALPHAD_df['alloy_string']]



In [11]:
# Create 5 CV groups by clustering similar alloys together
# Standardize features for clustering
kmeans_scaler = StandardScaler()
kmeans_X_scaled = kmeans_scaler.fit_transform(X_train_opt)

# Use KMeans to group similar alloys into 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cv_groups = kmeans.fit_predict(kmeans_X_scaled)

# Add group assignments to a dataframe for reference
cv_group_df = pd.DataFrame({
    'formula': train_opt_CALPHAD_df['alloy_string'],
    'cv_group': cv_groups
})

print("CV Group distribution:")
print(cv_group_df['cv_group'].value_counts().sort_index())
print(f"\nTotal samples: {len(cv_groups)}")
cv_group_df.head(10)

CV Group distribution:
cv_group
0    512
1    146
2     86
3     96
4     42
Name: count, dtype: int64

Total samples: 882


,formula,cv_group
0,B22.00Co4.00Fe68.00Y6.00,0
1,Cu47.00Nb11.00Ni8.00Si1.00Ti33.00,0
2,B6.00C15.00Co6.42Er0.75Fe57.82Mo14.00,2
3,Be24.00Cu12.00Fe8.00Nb8.00Zr48.00,0
4,Al10.00Ce60.00Cu20.00Ni10.00,3
5,Cu20.00Gd10.00Mg65.00Ni5.00,1
6,Ca55.00Cu20.00Mg25.00,1
7,Ag5.00Al12.50Cu15.00Fe5.00La62.50,3
8,B5.00C10.00Co35.00Fe40.00P10.00,2
9,Ca55.00Mg20.00Zn25.00,1


In [28]:

# --- Flexible NN for D_max regression (5760 → 1) ---
class DmaxNet(nn.Module):
    def __init__(self, input_dim, hidden_layers, dropout_rate=0.3, activation="relu"):
        """
        Args:
            input_dim:      number of input features (5760)
            hidden_layers:  list of ints, e.g. [1024, 512, 256]
            dropout_rate:   dropout probability applied after each hidden layer
            activation:     "relu", "leaky_relu", "elu", or "selu"
        """
        super().__init__()
        act_fn = {"relu": nn.ReLU, "leaky_relu": nn.LeakyReLU,
                  "elu": nn.ELU, "selu": nn.SELU}[activation]

        layers = []
        prev = input_dim
        for units in hidden_layers:
            layers.append(nn.Linear(prev, units))
            layers.append(nn.BatchNorm1d(units))
            layers.append(act_fn())
            layers.append(nn.Dropout(dropout_rate))
            prev = units
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train model for one epoch. Returns average training loss."""
    model.train()
    total_loss = 0.0
    n_batches = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


def evaluate(model, loader, criterion, device):
    """Evaluate model on a loader. Returns average loss."""
    model.eval()
    total_loss = 0.0
    n_batches = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            loss = criterion(model(xb), yb)
            total_loss += loss.item()
            n_batches += 1
    return total_loss / n_batches


def predict(model, loader, device):
    """Run inference on a loader. Returns (predictions, actuals) if labels exist, else just (predictions, None)."""
    model.eval()
    all_preds = []
    all_actuals = []
    has_labels = False
    with torch.no_grad():
        for batch in loader:
            if isinstance(batch, (list, tuple)) and len(batch) >= 2:
                xb, yb = batch[0], batch[1]
                has_labels = True
                all_actuals.append(yb.cpu().numpy())
            else:
                xb = batch[0] if isinstance(batch, (list, tuple)) else batch
            xb = xb.to(device)
            all_preds.append(model(xb).cpu().numpy())
    preds = np.concatenate(all_preds)
    actuals = np.concatenate(all_actuals) if has_labels else None
    return preds, actuals


In [29]:
#define the evaluate parameters function for the ax optimization loop of the dmax_nn model
def evaluate_parameters_NN_dmax(parameters):
    
    #break down the parameters
    batch_size = parameters.get("batch_size", 64)
    n_layers = parameters.get("n_layers", 3)
    layer_1_dim = parameters.get("layer_1_dim", 1024)
    layer_2_dim = parameters.get("layer_2_dim", 1024)
    layer_3_dim = parameters.get("layer_3_dim", 512)
    dropout_rate = parameters.get("dropout_rate", 0.3)
    activation = parameters.get("activation", "relu")
    lr = parameters.get("lr", 1e-3)
    weight_decay = parameters.get("weight_decay", 1e-4)
    patience = parameters.get("patience", 20)
    
    #combine layer dimensions into a list for the model
    hidden_layers = [layer_1_dim, layer_2_dim, layer_3_dim][:n_layers]
    
    # Auto-detect device
    if torch.cuda.is_available():
        device = torch.device('cuda')
    elif torch.backends.mps.is_available():
        device = torch.device('mps')
    else:
        device = torch.device('cpu')
    print(f"Using device: {device}")
    
    
    #copy the x and y data
    y_data = y_train_opt.copy()
    x_data = X_train_opt.copy()
    
    #initialize cross fold info
    all_fold_mse = []
    all_fold_rmse = []
    all_fold_mae = []
    
    for fold in range(1,6):
        print(f"Starting fold {fold}...")
        
        #Establish from the folds the train and test data
        X_fold_train = x_data[cv_groups != (fold-1)]
        y_fold_train = np.array(y_data)[cv_groups != (fold-1)]
        
        X_fold_test = x_data[cv_groups == (fold-1)]
        y_fold_test = np.array(y_data)[cv_groups == (fold-1)]
        
        #split the fold train data into train and validation data
        X_fold_train, X_fold_val, y_fold_train, y_fold_val = train_test_split(X_fold_train, y_fold_train, test_size=0.2, random_state=42)
        
        #scale the X_data
        Scaler_X_fold = StandardScaler()
        X_fold_train_scaled = Scaler_X_fold.fit_transform(X_fold_train)
        X_fold_val_scaled = Scaler_X_fold.transform(X_fold_val)
        X_fold_test_scaled = Scaler_X_fold.transform(X_fold_test)
        
        #Scale the y data
        Scaler_y_fold = StandardScaler()
        y_fold_train_scaled = Scaler_y_fold.fit_transform(y_fold_train.reshape(-1, 1)).flatten()
        y_fold_val_scaled = Scaler_y_fold.transform(y_fold_val.reshape(-1, 1)).flatten()
        y_fold_test_scaled = Scaler_y_fold.transform(y_fold_test.reshape(-1, 1)).flatten()
        
        #convert all data to tensors
        X_fold_train_tensor = torch.tensor(X_fold_train_scaled, dtype=torch.float32).to(device)
        y_fold_train_tensor = torch.tensor(y_fold_train_scaled, dtype=torch.float32).to(device)
        X_fold_val_tensor = torch.tensor(X_fold_val_scaled, dtype=torch.float32).to(device)
        y_fold_val_tensor = torch.tensor(y_fold_val_scaled, dtype=torch.float32).to(device)
        X_fold_test_tensor = torch.tensor(X_fold_test_scaled, dtype=torch.float32).to(device)
        y_fold_test_tensor = torch.tensor(y_fold_test_scaled, dtype=torch.float32).to(device)
        
        #convert tensors to datasets then dataloaders
        train_ds = TensorDataset(X_fold_train_tensor, y_fold_train_tensor)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(42))
        val_ds = TensorDataset(X_fold_val_tensor, y_fold_val_tensor)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=True, generator=torch.Generator().manual_seed(42)) 
        test_ds = TensorDataset(X_fold_test_tensor, y_fold_test_tensor)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, generator=torch.Generator().manual_seed(42))    

        #initialize the model
        input_dim = X_fold_train_tensor.shape[1]
        model = DmaxNet(input_dim, hidden_layers, dropout_rate, activation).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=patience // 3, factor=0.5)
        criterion = nn.MSELoss()
        
        
        best_val_loss = float("inf")
        best_state = None
        wait = 0
        
        for epoch in range(1000):

            train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
            val_loss = evaluate(model, val_loader, criterion, device)
            if epoch % 10 == 0:
                print(f"  Epoch {epoch}... Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
            scheduler.step(val_loss)
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
        
        # Restore best model and evaluate on fold test set
        model.load_state_dict(best_state)
        
        #predict the fold test set and inverse transform the predictions and actuals back to original scale
        y_fold_test_pred_scaled, y_fold_test_actual_scaled = predict(model, test_loader, device)
        y_fold_test_pred = Scaler_y_fold.inverse_transform(y_fold_test_pred_scaled.reshape(-1, 1)).flatten()
        y_fold_test_actual = Scaler_y_fold.inverse_transform(y_fold_test_actual_scaled.reshape(-1, 1)).flatten()
        
        #calculate the fold mse, rmse, and mae and add to the list of fold metrics
        fold_mse = np.mean((y_fold_test_pred - y_fold_test_actual) ** 2)
        fold_rmse = np.sqrt(fold_mse)
        fold_mae = np.mean(np.abs(y_fold_test_pred - y_fold_test_actual))
        
        all_fold_mse.append(fold_mse)
        all_fold_rmse.append(fold_rmse)
        all_fold_mae.append(fold_mae)
        
        #print the fold metrics
        print(f"Fold {fold} - MSE: {fold_mse:.4f}, RMSE: {fold_rmse:.4f}, MAE: {fold_mae:.4f}")

    mean_rmse = np.mean(all_fold_rmse)
    mean_mse = np.mean(all_fold_mse)
    mean_mae = np.mean(all_fold_mae)
    print(f"\nMean CV MSE: {mean_mse:.4f}, Mean CV RMSE: {mean_rmse:.4f}, Mean CV MAE: {mean_mae:.4f}")
    return mean_rmse, sem(all_fold_rmse)


In [30]:
#initialize the ax client for the dmax_nn model
ax_client_dmax_nn = AxClient()
ax_client_dmax_nn.create_experiment(
    name="NN CBVF + CALPHAD to D_max Regression",
    parameters=[
        {
            "name": "dropout_rate",
            "type": "range",
            "bounds": [0.0, 0.5],
        },
        {
            "name": "activation",
            "type": "choice",
            "values": ["relu", "leaky_relu", "elu", "selu", "gelu"],
            "is_ordered": False,
            "sort_values": False,
        },
        {
            "name": "weight_decay",
            "type": "range",
            "bounds": [1e-6, 1e-2],
            "log_scale": True,
        },
        {
            "name": "lr",
            "type": "range",
            "bounds": [1e-5, 1e-2],
            "log_scale": True,
        },
        {
            "name": "batch_size",
            "type": "choice",
            "values": [32, 64, 128, 256],
            "is_ordered": True,
            "sort_values": True,
        },
        {
            "name": "early_stopping_patience",
            "type": "range",
            "bounds": [10, 50],
            "value_type": "int",
        },
        {
            "name": "n_layers",
            "type": "choice",
            "values": [1, 2, 3],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,
        },
        {
            "name": "hidden1",
            "type": "choice",
            "values": [128, 256, 512, 1024, 2048],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,

        },
        {
            "name": "hidden2",
            "type": "choice",
            "values": [64, 128, 256, 512, 1024],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,

        },
        {
            "name": "hidden3",
            "type": "choice",
            "values": [32, 64, 128, 256, 512],
            "value_type": "int",
            "is_ordered": True,
            "sort_values": True,
        }
        
    ],
    objectives={"avg_rmse_nonzero": ObjectiveProperties(minimize=True)}
)

[INFO 04-01 13:22:04] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter dropout_rate. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-01 13:22:04] ax.service.utils.instantiation: Inferred value type of ParameterType.STRING for parameter activation. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-01 13:22:04] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter weight_decay. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.
[INFO 04-01 13:22:04] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter lr. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str'

In [ ]:
#Perform the ax optimization loop for the dmax_nn model

#initialize the save path for the dmax_nn ax client
ax_client_dmax_nn_save_path = r"Ax_checkpoints\ax_client_dmax_nn_checkpoint.json"

#check if there is an existing checkpoint and load it, otherwise start a new optimization loop
try:
    ax_client_dmax_nn = AxClient.load_from_json_file(filepath=ax_client_dmax_nn_save_path)
    print(f"Loaded existing Ax client checkpoint from {ax_client_dmax_nn_save_path}")
    
    completed_trials = len(ax_client_dmax_nn.experiment.trials)
    target_trials = 100
    remaining_trials = target_trials - completed_trials
    
    print(f"Loaded {completed_trials} completed trials from ax_client_dmax_nn.json")
    print(f"Remaining trials to reach {target_trials}: {remaining_trials}")
    
    for i in range(remaining_trials):
        parameters, trial_index = ax_client_dmax_nn.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_NN_dmax(parameters)
        ax_client_dmax_nn.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_dmax_nn.save_to_json_file(filepath=ax_client_dmax_nn_save_path)
        print(f"Saved Ax client checkpoint")
    
    
except Exception as e:
    print(f"No existing checkpoint found, starting new optimization loop. Error: {e}")
    
    for i in range(20):
        parameters, trial_index = ax_client_dmax_nn.get_next_trial()
        print(f"\nStarting trial {trial_index} with parameters: {parameters}")
        mean_rmse, rmse_sem = evaluate_parameters_NN_dmax(parameters)
        ax_client_dmax_nn.complete_trial(trial_index=trial_index, raw_data={"avg_rmse_nonzero": (mean_rmse, rmse_sem)})
        print(f"Completed trial {trial_index} with mean RMSE: {mean_rmse:.4f} ± {rmse_sem:.4f}")
        
        # Save the Ax client state after each trial
        ax_client_dmax_nn.save_to_json_file(filepath=ax_client_dmax_nn_save_path)
        print(f"Saved Ax client checkpoint")

[INFO 04-01 13:22:05] ax.service.ax_client: Generated new trial 0 with parameters {'dropout_rate': 0.245605, 'weight_decay': 0.001416, 'lr': 2.7e-05, 'batch_size': 256, 'early_stopping_patience': 42, 'n_layers': 2, 'hidden1': 128, 'hidden2': 1024, 'hidden3': 512, 'activation': 'elu'} using model Sobol.


No existing checkpoint found, starting new optimization loop. Error: [Errno 2] No such file or directory: 'Ax_checkpoints\\ax_client_dmax_nn_checkpoint.json'

Starting trial 0 with parameters: {'dropout_rate': 0.2456045001745224, 'weight_decay': 0.0014159011179192306, 'lr': 2.681057802282944e-05, 'batch_size': 256, 'early_stopping_patience': 42, 'n_layers': 2, 'hidden1': 128, 'hidden2': 1024, 'hidden3': 512, 'activation': 'elu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 1.4691, Val Loss: 0.8433
  Epoch 10... Train Loss: 0.5297, Val Loss: 0.4475
  Epoch 20... Train Loss: 0.4628, Val Loss: 0.3348
  Epoch 30... Train Loss: 0.3835, Val Loss: 0.4845
  Epoch 40... Train Loss: 0.4104, Val Loss: 0.4678
Fold 1 - MSE: 40.5950, RMSE: 6.3714, MAE: 4.3237
Starting fold 2...
  Epoch 0... Train Loss: 1.0637, Val Loss: 0.8938
  Epoch 10... Train Loss: 0.4860, Val Loss: 0.6619
  Epoch 20... Train Loss: 0.4050, Val Loss: 0.6956
Fold 2 - MSE: 37.8106, RMSE: 6.1490, MAE: 3.7680
Starti

[INFO 04-01 13:22:10] ax.service.ax_client: Completed trial 0 with data: {'avg_rmse_nonzero': 6.757413}.
[INFO 04-01 13:22:10] ax.service.ax_client: Generated new trial 1 with parameters {'dropout_rate': 0.333383, 'weight_decay': 2.5e-05, 'lr': 0.009614, 'batch_size': 64, 'early_stopping_patience': 10, 'n_layers': 1, 'hidden1': 2048, 'hidden2': 64, 'hidden3': 32, 'activation': 'leaky_relu'} using model Sobol.


  Epoch 20... Train Loss: 0.4026, Val Loss: 1.0067
Fold 5 - MSE: 48.8130, RMSE: 6.9866, MAE: 6.3008

Mean CV MSE: 46.5161, Mean CV RMSE: 6.7574, Mean CV MAE: 4.8279
Completed trial 0 with mean RMSE: 6.7574 ± 0.4619
Saved Ax client checkpoint

Starting trial 1 with parameters: {'dropout_rate': 0.33338316017761827, 'weight_decay': 2.451309215924434e-05, 'lr': 0.009614034981393705, 'batch_size': 64, 'early_stopping_patience': 10, 'n_layers': 1, 'hidden1': 2048, 'hidden2': 64, 'hidden3': 32, 'activation': 'leaky_relu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 3.5392, Val Loss: 10.6551
  Epoch 10... Train Loss: 0.5322, Val Loss: 0.5504
  Epoch 20... Train Loss: 0.4732, Val Loss: 0.3832
  Epoch 30... Train Loss: 0.3941, Val Loss: 0.2512
  Epoch 40... Train Loss: 0.3494, Val Loss: 0.2582
  Epoch 50... Train Loss: 0.3386, Val Loss: 0.3524
  Epoch 60... Train Loss: 0.3431, Val Loss: 0.2584
  Epoch 70... Train Loss: 0.3161, Val Loss: 0.3181
Fold 1 - MSE: 37.6197, RMSE: 6.13

[INFO 04-01 13:22:23] ax.service.ax_client: Completed trial 1 with data: {'avg_rmse_nonzero': 6.263437}.
[INFO 04-01 13:22:24] ax.service.ax_client: Generated new trial 2 with parameters {'dropout_rate': 0.485469, 'weight_decay': 0.000289, 'lr': 0.000969, 'batch_size': 32, 'early_stopping_patience': 26, 'n_layers': 1, 'hidden1': 1024, 'hidden2': 256, 'hidden3': 256, 'activation': 'elu'} using model Sobol.


  Epoch 60... Train Loss: 0.2582, Val Loss: 0.7277
Fold 5 - MSE: 9.6959, RMSE: 3.1138, MAE: 1.8027

Mean CV MSE: 42.7042, Mean CV RMSE: 6.2634, Mean CV MAE: 4.2394
Completed trial 1 with mean RMSE: 6.2634 ± 0.9319
Saved Ax client checkpoint

Starting trial 2 with parameters: {'dropout_rate': 0.485468870960176, 'weight_decay': 0.0002894903156975929, 'lr': 0.0009688950028256467, 'batch_size': 32, 'early_stopping_patience': 26, 'n_layers': 1, 'hidden1': 1024, 'hidden2': 256, 'hidden3': 256, 'activation': 'elu'}
Using device: cuda
Starting fold 1...
  Epoch 0... Train Loss: 8.0354, Val Loss: 2.0908
  Epoch 10... Train Loss: 0.4557, Val Loss: 0.3924
  Epoch 20... Train Loss: 0.5127, Val Loss: 0.4693
  Epoch 30... Train Loss: 0.4165, Val Loss: 0.3743
Fold 1 - MSE: 45.9376, RMSE: 6.7777, MAE: 4.5797
Starting fold 2...
  Epoch 0... Train Loss: 5.8997, Val Loss: 1.8517
  Epoch 10... Train Loss: 0.4936, Val Loss: 0.6644
  Epoch 20... Train Loss: 0.3848, Val Loss: 0.6114
  Epoch 30... Train Loss: